In [53]:
#!unzip -q "/content/C-NMC 2019 (PKG).zip"

#!pip install scikit-learn torchvision

import copy
import numpy as np
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import f1_score


BASE_DIR = Path("/content")
DATA_ROOT = Path("/content/C-NMC 2019 (PKG)/C-NMC_training_data")
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

BATCH_SIZE = 32
NUM_EPOCHS = 10
PATIENCE = 3
LR_HEAD = 1e-3
LR_FINE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])




def load_datasets():
    train_folds = []
    val_folds = []
    for fold in ["fold_0", "fold_1", "fold_2"]:
        fold_path = DATA_ROOT / fold
        train_folds.append(datasets.ImageFolder(fold_path, transform=train_transform))
        val_folds.append(datasets.ImageFolder(fold_path, transform=val_transform))

    full_train_ds = ConcatDataset(train_folds)
    full_val_ds = ConcatDataset(val_folds)


    indices = torch.randperm(len(full_train_ds), generator=torch.Generator().manual_seed(42)).tolist()
    val_size = int(0.2 * len(full_train_ds))

    train_indices = indices[val_size:]
    val_indices = indices[:val_size]

    train_dataset = Subset(full_train_ds, train_indices)
    val_dataset = Subset(full_val_ds, val_indices)

    return train_dataset, val_dataset


def compute_class_weights(subset):

    concat_ds = subset.dataset
    all_labels = []
    for ds in concat_ds.datasets:
        all_labels.extend(ds.targets)

    subset_labels = [all_labels[i] for i in subset.indices]
    counts = Counter(subset_labels)
    total = sum(counts.values())
    weights = [total / counts[i] for i in sorted(counts)]
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)


def build_model(name):
    if name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 2)
    else:
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, 2)
    return model.to(DEVICE)


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    preds_all, labels_all = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        # Mixed Precision Forward Pass
        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        preds_all.extend(torch.argmax(out, 1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())

    return f1_score(labels_all, preds_all, average="weighted")


def validate(model, loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.cuda.amp.autocast():
                out = model(x)

            preds_all.extend(torch.argmax(out, 1).cpu().numpy())
            labels_all.extend(y.cpu().numpy())

    return f1_score(labels_all, preds_all, average="weighted")


def train_model(name):
    print(f"\n🚀 Initializing {name} on {DEVICE}...")
    train_dataset, val_dataset = load_datasets()

    # Optimized Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    weights = compute_class_weights(train_dataset)
    criterion = nn.CrossEntropyLoss(weight=weights)
    scaler = torch.cuda.amp.GradScaler()  # For AMP
    model = build_model(name)

    # ========= PHASE 1: Warmup Head =========
    for p in model.parameters(): p.requires_grad = False
    head_params = model.fc.parameters() if name == "resnet50" else model.classifier[6].parameters()
    for p in head_params: p.requires_grad = True

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)

    print("Phase 1: Training Head...")
    for epoch in range(2):
        _ = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_f1 = validate(model, val_loader)
        print(f"  [HEAD] Epoch {epoch + 1} Val F1: {val_f1:.4f}")

    # ========= PHASE 2: Fine-Tuning =========
    for p in model.parameters(): p.requires_grad = True
    optimizer = optim.Adam(model.parameters(), lr=LR_FINE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=1)

    best_f1 = 0
    patience_counter = 0
    best_model_wts = copy.deepcopy(model.state_dict())

    print("Phase 2: Fine-Tuning Backbone...")
    for epoch in range(NUM_EPOCHS):
        _ = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_f1 = validate(model, val_loader)
        scheduler.step(val_f1)

        print(f"  [FT] Epoch {epoch + 1} Val F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print("  [INFO] Early stopping triggered.")
            break

    torch.save(best_model_wts, OUTPUT_DIR / f"{name}_best.pth")
    print(f"✅ Finished {name}. Best Val F1: {best_f1:.4f}")



if __name__ == "__main__":
    train_model("resnet50")
    train_model("vgg16")


🚀 Initializing resnet50 on cuda...
Phase 1: Training Head...
  [HEAD] Epoch 1 Val F1: 0.7063
  [HEAD] Epoch 2 Val F1: 0.7212
Phase 2: Fine-Tuning Backbone...
  [FT] Epoch 1 Val F1: 0.8771
  [FT] Epoch 2 Val F1: 0.8510
  [FT] Epoch 3 Val F1: 0.9087
  [FT] Epoch 4 Val F1: 0.9172
  [FT] Epoch 5 Val F1: 0.9243
  [FT] Epoch 6 Val F1: 0.9073
  [FT] Epoch 7 Val F1: 0.8567
  [FT] Epoch 8 Val F1: 0.9286
  [FT] Epoch 9 Val F1: 0.9389
  [FT] Epoch 10 Val F1: 0.9421
✅ Finished resnet50. Best Val F1: 0.9421

🚀 Initializing vgg16 on cuda...
Phase 1: Training Head...
  [HEAD] Epoch 1 Val F1: 0.6102
  [HEAD] Epoch 2 Val F1: 0.5826
Phase 2: Fine-Tuning Backbone...
  [FT] Epoch 1 Val F1: 0.8125
  [FT] Epoch 2 Val F1: 0.8907
  [FT] Epoch 3 Val F1: 0.7094
  [FT] Epoch 4 Val F1: 0.8871
  [FT] Epoch 5 Val F1: 0.9028
  [FT] Epoch 6 Val F1: 0.9087
  [FT] Epoch 7 Val F1: 0.9158
  [FT] Epoch 8 Val F1: 0.8918
  [FT] Epoch 9 Val F1: 0.9190
  [FT] Epoch 10 Val F1: 0.9243
✅ Finished vgg16. Best Val F1: 0.9243
